# Compare a small metadata network: deduplicated Subset data

Can a small neural network improve on the spline using age, sex and anatomical site, the metadata recorded alongside each image? We also check how repeated training changes the result.

[05_metadata_models.ipynb](05_metadata_models.ipynb) supplies the fixed spline reference. Both methods use the same patient folds, meaning groups of patients, and are checked on people excluded from their own training.

The verified outputs below compare the fixed three-network average with the spline and report every prescribed seed. Primary mean-fold ROC-AUC and secondary pooled scores are distinguished explicitly. The computed findings, rather than historical numbers in prose, determine the observed comparison. The network reads no image pixels.

For metric definitions, see the [short concept guide](../concepts.md).


**Scope and preservation:** this notebook uses the complete archived original as its section and visual template. Numerical findings are computed from the explicitly identified versioned run. The original experiments remain archived; no reserved-test labels or final all-development fit are used here.

**Experiment-scope change: version 2.** The completed AP-selected version 1 is preserved. This separately declared experiment selects settings by mean inner-fold ROC-AUC and reports mean outer-fold ROC-AUC as primary. AP, log loss and Brier are secondary; pooled summaries are supporting. The prior report’s heading structure, model colours, fonts and explanatory detail are retained. No all-development model or reserved-test result is produced.

## Setup and data

Use the frozen `dedup_v1` development manifest with the project Python kernel. The notebook finds the project root from its current folder. The reserved test is not loaded.

Before these runs, fix one hidden layer with 32 units, 20% dropout, training settings, seeds 42, 142 and 242, and equal averaging. A seed makes random choices repeatable. Report every run and their average rather than selecting the highest score.

The reused patient folds have guided earlier comparisons, so these remain development results.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src/melanoma_detection").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from melanoma_detection.metadata_tuning_v2_reporting import (
    load_tuning, load_baseline_report, render_section, render_baseline_section,
    display_historical_cell, display_historical_source, original_snapshot,
)
plt.rcdefaults()
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120})
ORIGINAL_EXPORT_DPI = 150
NN_COMPARISON_COLORS = ['#2563EB', '#D97706', '#D97706', '#D97706', '#303030']
report = load_tuning(PROJECT_ROOT, 'dedup_v1', run_id='roc_auc_grid36_v1')
report["export_dpi"] = ORIGINAL_EXPORT_DPI
display(Markdown("**Version-2 saved-results report:** " + report["status"] +
    ". ROC-AUC is primary; AP and probability errors are secondary. No training, final fitting or reserved-test loading starts here."))


### 1. Check the inputs in the saved experiment

Validate target text before conversion: only exact labels 0 (benign) and 1 (malignant) are accepted. Otherwise rounding or integer conversion could hide a bad value such as `1.5`. Each image must occur once, every patient must stay in one fold, and each fold must contain both classes.

Missing age is filled with the training median. Missing sex or site gets a fixed `Missing` label. Category columns are learned separately within training, keeping evaluation patients outside preprocessing.


In [ ]:
render_section(report, 'inputs')

## How the network is trained

### Use a small network for tabular information

Turn age, sex and site into numerical inputs. Age uses the training median for missing values and training-based scaling. One-hot encoding gives each category a yes/no column. These are a short list of numbers, rather than image pixels, so the network learns from scratch.

Its hidden layer has 32 units, each combining the inputs with learned weights. ReLU keeps positive activations and replaces negative ones with zero, allowing flexible relationships. The final sigmoid maps the output to [0,1]. This is a probability estimate; calibration, agreement with observed rates, still needs checking.

Dropout masks 20% of hidden activations during training and is disabled when predicting. Weight decay discourages large weights. These limit overfitting, learning details that fail on new patients. With 422 patients and only 127 malignant images, a small fixed network is a cautious comparison, rather than a claim of the best design.

### Choose training length without scoring patients

An epoch is one pass through training records. Binary cross-entropy measures probability error; smaller is better.

For each outer fold and run:

1. Set aside the outer patients for scoring.
2. Use one of the four remaining folds to choose the best epoch. Fit preprocessing and the network on the other three.
3. Start fresh with the same seed, refit preprocessing on all four training folds, and train for that epoch count.
4. Predict the outer fold once. Repeat until each image has one held-out prediction per run, called out-of-fold (OOF) prediction.

The stopping fold follows `(outer fold + 1 + run index) % 5`. Stop after 12 epochs without meaningful improvement once 10 epochs have been attempted, or at 100 epochs. The selected best earlier epoch can be below 10. Outer labels never choose it.

### Repeat a fixed procedure three times

Initial weights, batch order and dropout are random. The stopping group also changes across runs, so their spread reflects both choices together. All runs reuse the same patients: their standard deviation describes training variation, not a confidence interval for new-patient performance. Keep every run and their equal average. Training gives malignant cases no extra loss weight.


In [ ]:
render_section(report, 'nn_procedure')

In [ ]:
display(Markdown("The separate runner owns model fitting. This report reads completed version-2 outputs only."))

### 2. Review the prescribed runs on five patient-held-out folds

The checks below explicitly enforce patient separation and exactly one prediction per image per run. Preprocessing is fitted again whenever the training patients change. The spline reference is fitted once per outer fold because its fixed procedure is deterministic and does not change across network seeds.

This cell verifies and reads saved experiment outputs. Training is performed separately by the versioned runner; executing this report starts no fits.

In [ ]:
render_section(report, 'nn_runs')

### 3. Read the performance and variation comparison

The primary score averages the five outer-fold scores equally. A separate supporting chart pools all held-out predictions; these aggregations can differ because cross-fold score scales and malignant-image counts differ. Average precision (AP), the primary measure, rewards concentrating malignant images near the top of the ranking. ROC-AUC measures how often a malignant image scores above a benign one; 0.5 is chance. Higher is better for both. Brier score measures squared probability error; lower is better, but a small value alone does not establish accurate risks.

In the charts, the horizontal axis lists models and runs: blue spline, orange individual networks, dark-grey average. The vertical axis is ROC-AUC on the left and AP on the right. The dashed AP line is the malignant-image share; annotations give exact scores. No sampling uncertainty bars are shown.

The variation table describes the three individual runs, whose seed and stopping fold both change. Compare their spread and average with the spline. A higher ROC-AUC does not by itself overturn the comparison on AP, the chosen primary measure. Read the computed findings below.


In [ ]:
render_section(report, 'nn_metrics')

### Read the recorded training duration and histories

The epoch choices are made on training-only stopping groups. Orange marks show the individual fixed-NN runs; charcoal shows their median. Loss curves, when recorded, show fit and stopping probability error. They describe optimization, not performance on the outer patients.

In [ ]:
render_section(report, 'nn_epochs')

### 4. Read predictions and run settings

Save one row per image so later checks compare identical patients. SHA-256 fingerprints identify the exact input, prediction and executable-code bytes; markdown and saved outputs do not affect the code fingerprint. Settings record preprocessing, network design, training and package versions. Fingerprints support traceability, rather than proving correctness.

These are development outputs. Reports read hash-verified versioned predictions and settings and write presentation copies only inside the new run's `report/` directory. Historical `d1` results and the previous notebook are preserved in the archive.

In [ ]:
render_section(report, 'run_settings')

## Limitations

This stage compares one prescribed small architecture. Stage 07 separately searches hidden-layer size, learning rate and dropout through nested patient checks; even that bounded search cannot identify the best possible network design. Three fits check sensitivity to a few training choices. Because both the seed and stopping fold change together, their spread cannot isolate the effect of either choice.

All runs reuse the same patients. They are not three independent datasets, and the variability table is not a sampling confidence interval. This notebook does not establish calibrated probabilities or reliable performance across demographic groups; those checks belong in notebook 08.

The patient folds have guided earlier comparisons. Keeping outer patients outside fitting and epoch selection prevents direct leakage, but repeated model choices based on these folds can still make the chosen method look optimistic. Metadata also contains no lesion appearance. The reserved test is not used to evaluate these models, and performance on another hospital's patients or clinical safety is not established.


## Summary

The computed summary below reports both aggregations and the observed NN-minus-spline difference. Keep all prescribed seeds and their equal average in view: choosing the best individual run after seeing its score could favour a lucky result. Paired uncertainty and the simpler model's practical advantages remain relevant alongside the ranking scores.

Next, [07_metadata_tuning.ipynb](07_metadata_tuning.ipynb) checks model settings and [08_metadata_robustness.ipynb](08_metadata_robustness.ipynb) adds patient-based intervals, calibration and subgroup checks. Freeze the complete method before final testing.


In [ ]:
render_section(report, 'nn_summary')